In [10]:
packages <- c("readr", "jsonlite", "readxl", "writexl", "dplyr",
              "lubridate", "DBI", "RSQLite", "stringr")
new_pkgs <- packages[!(packages %in% installed.packages()[, "Package"])]
if (length(new_pkgs)) install.packages(new_pkgs)

library(readr)
library(jsonlite)
library(readxl)
library(writexl)
library(dplyr)
library(lubridate)
library(DBI)
library(RSQLite)
library(stringr)

In [11]:
uci_url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
raw_path <- "OnlineRetail.xlsx"

if (!file.exists(raw_path)) {
  download.file(uci_url, destfile = raw_path, mode = "wb")
}

raw <- read_excel(raw_path)
names(raw) <- make.names(names(raw))

transactions_raw <- raw %>%
  select(InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate)

products_raw <- raw %>%
  select(StockCode, Description, UnitPrice) %>%
  distinct(StockCode, .keep_all = TRUE)

customers_raw <- raw %>%
  select(CustomerID, Country) %>%
  filter(!is.na(CustomerID)) %>%
  distinct(CustomerID, .keep_all = TRUE)

write_csv(transactions_raw, "transactions.csv")
write_json(products_raw, "products.json")
write_xlsx(customers_raw, "customers.xlsx")

In [12]:
transactions <- read_csv("transactions.csv", show_col_types = FALSE)
products     <- fromJSON("products.json") %>% as_tibble()
customers    <- read_excel("customers.xlsx")

glimpse(transactions)
glimpse(products)
glimpse(customers)

sum(is.na(transactions))
sum(duplicated(transactions))

Rows: 541,909
Columns: 5
$ InvoiceNo   <chr> "536365", "536365", "536365", "536365", "536365", "536365"…
$ StockCode   <chr> "85123A", "71053", "84406B", "84029G", "84029E", "22752", …
$ CustomerID  <dbl> 17850, 17850, 17850, 17850, 17850, 17850, 17850, 17850, 17…
$ Quantity    <dbl> 6, 6, 8, 6, 6, 2, 6, 6, 6, 32, 6, 6, 8, 6, 6, 3, 2, 3, 3, …
$ InvoiceDate <dttm> 2010-12-01 08:26:00, 2010-12-01 08:26:00, 2010-12-01 08:2…
Rows: 4,070
Columns: 3
$ StockCode   <chr> "85123A", "71053", "84406B", "84029G", "84029E", "22752", …
$ Description <chr> "WHITE HANGING HEART T-LIGHT HOLDER", "WHITE METAL LANTERN…
$ UnitPrice   <dbl> 2.55, 3.39, 2.75, 3.39, 3.39, 7.65, 4.25, 1.85, 1.85, 1.69…
Rows: 4,372
Columns: 2
$ CustomerID <dbl> 17850, 13047, 12583, 13748, 15100, 15291, 14688, 17809, 153…
$ Country    <chr> "United Kingdom", "United Kingdom", "France", "United Kingd…


[1] 135080

[1] 5429

In [13]:
# Cleaning decisions:
# - Drop rows missing CustomerID: can't attribute revenue without one
# - Drop exact duplicate rows
# - Drop Quantity <= 0: returns/cancellations, would distort revenue
# - Drop UnitPrice <= 0: data errors, not real sales

transactions_clean <- transactions %>%
  filter(!is.na(CustomerID)) %>%
  distinct() %>%
  filter(Quantity > 0) %>%
  mutate(InvoiceDate = as.POSIXct(InvoiceDate, format = "%Y-%m-%d %H:%M:%S"))

products_clean <- products %>%
  filter(!is.na(UnitPrice), UnitPrice > 0) %>%
  distinct(StockCode, .keep_all = TRUE)

customers_clean <- customers %>%
  filter(!is.na(CustomerID), !is.na(Country)) %>%
  distinct(CustomerID, .keep_all = TRUE)

cat("Rows before cleaning:", nrow(transactions), "\n")
cat("Rows after cleaning:", nrow(transactions_clean), "\n")

Rows before cleaning: 541909 
Rows after cleaning: 392708 


In [14]:
integrated <- transactions_clean %>%
  inner_join(products_clean, by = "StockCode") %>%
  left_join(customers_clean, by = "CustomerID") %>%
  mutate(Revenue = Quantity * UnitPrice)

cat("Final integrated dataset dimensions:", dim(integrated), "\n")

unmatched_products <- anti_join(transactions_clean, products_clean, by = "StockCode")
unmatched_customers <- anti_join(transactions_clean, customers_clean, by = "CustomerID")
cat("Transactions with no matching product:", nrow(unmatched_products), "\n")
cat("Transactions with no matching customer record:", nrow(unmatched_customers), "\n")

# Join justification:
# - inner_join on products: no UnitPrice = no Revenue = unusable row
# - left_join on customers: every cleaned txn already has a CustomerID,
#   so this just attaches Country without dropping anything

Final integrated dataset dimensions: 387877 9 
Transactions with no matching product: 4831 
Transactions with no matching customer record: 0 


In [15]:
total_revenue <- sum(integrated$Revenue)
cat("Total Sales Revenue:", total_revenue, "\n")

top5_products <- integrated %>%
  group_by(StockCode, Description) %>%
  summarise(TotalRevenue = sum(Revenue), .groups = "drop") %>%
  arrange(desc(TotalRevenue)) %>%
  slice_head(n = 5)
print(top5_products)

top5_countries <- integrated %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue), .groups = "drop") %>%
  arrange(desc(TotalRevenue)) %>%
  slice_head(n = 5)
print(top5_countries)

top5_customers <- integrated %>%
  group_by(CustomerID) %>%
  summarise(TotalSpend = sum(Revenue), .groups = "drop") %>%
  arrange(desc(TotalSpend)) %>%
  slice_head(n = 5)
print(top5_customers)

Total Sales Revenue: 10752840 
# A tibble: 5 × 3
  StockCode Description                        TotalRevenue
  <chr>     <chr>                                     <dbl>
1 23843     PAPER CRAFT , LITTLE BIRDIE             168470.
2 47566     PARTY BUNTING                           142438.
3 22423     REGENCY CAKESTAND 3 TIER                135605.
4 85123A    WHITE HANGING HEART T-LIGHT HOLDER       93746.
5 23166     MEDIUM CERAMIC TOP STORAGE JAR           81033.
# A tibble: 5 × 2
  Country        TotalRevenue
  <chr>                 <dbl>
1 United Kingdom     8861857.
2 Netherlands         363884.
3 EIRE                331660.
4 Germany             263819.
5 France              226976.
# A tibble: 5 × 2
  CustomerID TotalSpend
       <dbl>      <dbl>
1      18102    408760.
2      14646    357531.
3      17450    186038.
4      14911    182691.
5      16446    168472.


In [16]:
customer_value <- integrated %>%
  group_by(CustomerID) %>%
  summarise(TotalSpend = sum(Revenue), .groups = "drop") %>%
  mutate(ValueSegment = case_when(
    TotalSpend >= 5000 ~ "Premium",
    TotalSpend >= 1000 ~ "High Value",
    TotalSpend >= 200   ~ "Medium Value",
    TRUE                 ~ "Low Value"
  ))

table(customer_value$ValueSegment)
# Check summary(customer_value$TotalSpend) and adjust thresholds
# above if your data's spread looks different


  High Value    Low Value Medium Value      Premium 
        1542          499         1949          349 

In [17]:
print(top5_countries)   # highest -> high-performing market

lowest_country <- integrated %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue), .groups = "drop") %>%
  arrange(TotalRevenue) %>%
  slice_head(n = 1)
print(lowest_country)   # lowest -> underperforming market

# A tibble: 5 × 2
  Country        TotalRevenue
  <chr>                 <dbl>
1 United Kingdom     8861857.
2 Netherlands         363884.
3 EIRE                331660.
4 Germany             263819.
5 France              226976.
# A tibble: 1 × 2
  Country      TotalRevenue
  <chr>               <dbl>
1 Saudi Arabia         181.


In [18]:
con <- dbConnect(RSQLite::SQLite(), "retail_analysis.sqlite")

dbWriteTable(con, "retail_sales", integrated, overwrite = TRUE)

q1 <- dbGetQuery(con, "
  SELECT CustomerID, SUM(Revenue) AS TotalRevenue
  FROM retail_sales
  GROUP BY CustomerID
  ORDER BY TotalRevenue DESC
  LIMIT 5;
")
print(q1)

q2 <- dbGetQuery(con, "
  SELECT Country, SUM(Revenue) AS TotalRevenue
  FROM retail_sales
  GROUP BY Country
  ORDER BY TotalRevenue DESC;
")
print(q2)

dbDisconnect(con)

  CustomerID TotalRevenue
1      18102     408760.0
2      14646     357531.1
3      17450     186038.0
4      14911     182690.5
5      16446     168472.5
                Country TotalRevenue
1        United Kingdom   8861857.13
2           Netherlands    363884.48
3                  EIRE    331660.17
4               Germany    263818.97
5                France    226975.60
6             Australia    173918.61
7                 Spain     67426.09
8           Switzerland     66619.97
9                 Japan     48600.22
10              Belgium     47858.02
11               Sweden     43652.09
12               Norway     40283.20
13             Portugal     32679.45
14              Finland     23306.79
15      Channel Islands     22059.72
16              Denmark     21291.79
17                Italy     21065.45
18               Cyprus     16829.62
19            Singapore     11497.98
20               Poland     10264.54
21              Austria      9883.01
22               Israel      9